In [ ]:
#from eto import ETo, datasets
import pandas as pd
import os
import numpy as np
import csv

In [ ]:
from eto import ETo, datasets

In [ ]:
# repository root (notebooks live in <repo>/notebooks)
base_loc = os.path.abspath(os.path.join(os.getcwd(), '..'))
base_loc = os.path.join(base_loc, 'data')

In [ ]:
data_loc=os.path.join(base_loc,'weather')
aqua_loc=os.path.join(base_loc,'weather_aquacrop')

In [ ]:
def et_input(fn):
    df=pd.read_csv(fn,skiprows=14)
    df['PRECTOTCORR']=np.where(df['PRECTOTCORR']==-999,np.nan,df['PRECTOTCORR'])
    df['date']=df['YEAR'].astype(str)+'/'+df['DOY'].astype(str)
    df['date']=pd.to_datetime(df['date'],format='%Y/%j')
    df=df.rename(columns={'ALLSKY_SFC_SW_DWN':'R_s', 'T2M_MAX':'T_max', 'T2M_MIN':'T_min', 'RH2M':'RH_mean',
            'WS2M':'U_z'})
    et_data=df[['date' ,'R_s', 'T_max', 'T_min', 'RH_mean','U_z']]
    et_data=et_data.set_index('date')
    return et_data

In [ ]:
def et_param(fn):
    with open(fn, mode='r') as csv_file:
        csv_reader = csv.DictReader(csv_file)
        line_count = 0
        for row in csv_reader:
            if line_count <15:
                x=(row['-BEGIN HEADER-'])
                if 'Latitude' in x:
                    lat=(float(x[x.find('Latitude')+8:x.find('Latitude')+8+10]))
                if 'Longitude' in x:
                    lon=(float(x[x.find('Longitude')+9:]))
                if 'Elevation' in x:
                    z_msl=(float(x.split('=')[-1].split('me')[0]))
                line_count+=1
    return z_msl,lat,lon

In [ ]:
def aqua_output(fn,eto1,name):
    df=pd.read_csv(fn,skiprows=14)
    df['PRECTOTCORR']=np.where(df['PRECTOTCORR']==-999,0,df['PRECTOTCORR'])
    df['date']=df['YEAR'].astype(str)+'/'+df['DOY'].astype(str)
    df['date']=pd.to_datetime(df['date'],format='%Y/%j')
    df['Day']=df['date'].apply(lambda x:x.day)
    df['Month']=df['date'].apply(lambda x:x.month)
    df['Year']=df['date'].apply(lambda x:x.year)
    test=pd.merge(df,eto1,on='date')
    final=test[['Day', 'Month', 'Year', 'T2M_MIN','T2M_MAX','PRECTOTCORR', 'ETo_FAO_mm']]
    final=final.rename(columns={'T2M_MIN':'Tmin(C)', 'T2M_MAX':'Tmax(C)', 'PRECTOTCORR':'Prcp(mm)',
                                'ETo_FAO_mm':'Et0(mm)'})
    text=os.path.join(aqua_loc,name+'.txt')
    final.to_csv(text, sep=' ',index=False)
    return name

In [ ]:
for r,d,f in os.walk(data_loc):
    for fl in f:
        if fl.endswith('.csv'):
            fn=os.path.join(r,fl)
            et1 = ETo()
            tsdata = et_input(fn)
            freq='D'
            z_msl, lat, lon = et_param(fn)
            et1.param_est(tsdata, freq, z_msl, lat, lon)
            eto1 = et1.eto_fao()
            name=(fl.split('.')[0])
            print(aqua_output(fn,eto1,name))